# MP26 — Attention-Based SR Network

An image-level residual network predicts a correction to bicubic interpolation. Attention changes the features used to predict that correction. This lesson connects the inspectable gates from MP23–25 with actual synthetic super-resolution experiments. The compact model is not RCAN or an exact CBAM reproduction.

## Mathematics, dimensions and worked example

Input y is N×4×H×W with RGB and NIR in fixed stored DN/10000 units. A 3×3 head maps four bands to sixteen feature channels. The local path has two 3×3 convolutions with a ReLU between them. Channel then spatial attention weights that local feature tensor. The tail predicts 4×2²=16 channels, then PixelShuffle rearranges them to N×4×2H×2W. Bicubic B(y) is added at the image output: x_hat=B(y)+tail(A(local(head(y)))).

The L1 training loss is mean absolute difference over batch, channels and spatial positions. If an HR value is 0.7 and bicubic predicts 0.6, the desired residual is +0.1. Zeroing the residual tail gives exactly bicubic, including its possible interpolation overshoot; do not silently clip evaluation values.

For width16, channel attention adds (16×4+4)+(4×16+16)=148 parameters and spatial attention adds 2×7×7+1=99, totaling247. Learned features need not correspond individually to sensor bands. The reconstruction skip is image-level; this small network is not a deep stack of residual attention blocks.


In [ ]:
from pathlib import Path
import sys,json,torch
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/rssr').is_dir())
sys.path.insert(0,str(ROOT/'src'))
from mp26_tools import SRModel
torch.manual_seed(26)
x=torch.rand(1,4,5,7)
cnn=SRModel(bands=4,kind='cnn',zero_residual=True)
attention=SRModel(bands=4,kind='attention',zero_residual=True)
baseline=torch.nn.functional.interpolate(x,scale_factor=2,mode='bicubic',align_corners=False)
torch.testing.assert_close(attention(x),baseline)
assert attention(x).shape==(1,4,10,14)
counts=[sum(p.numel() for p in m.parameters()) for m in [cnn,attention]]
assert counts==[7552,7799] and counts[1]-counts[0]==247
print('CNN/attention parameters',counts)


## Reproducible comparison

The completed development study used four training and four validation NAIP scenes, split geographically before extracting sixteen nonoverlapping 32×32 HR patches per scene. NAIP/USDA/OpenSR source and checksums are recorded in dataset manifests. Values are stored DN/10000, not verified reflectance. LR is Gaussian sigma0.8 followed by area x2 downsampling. CPU float32; seeds17,23,41; Adam0.001; batch4; 1000 updates; L1; no augmentation or scheduler. Validation L1 every100 updates selects the checkpoint. Metrics use range1, border2 and equal scene weighting; validation supports model development, not independent test claims.

Run `python scripts/attention_sr_evidence.py` to export the already recorded training curves and table, requiring the existing run records. The notebook itself reads the portable committed evidence and does not train or access held-out data. A fresh experiment uses `scripts/submission_study.py`; follow its resumable manifest/checkpoint checks and resource budget. Do not rerun training to obtain a preferred result.

Mean validation gains over bicubic were +1.5709 dB for CNN and +1.7947 dB for attention. Seed standard deviations were0.2818 and0.1873 dB; these are not geographic confidence intervals. The attention overhead is247 parameters. The sampled CPU timing happened to favor attention in this run; extra operations do not imply a reliable speedup, and a separate controlled repeated benchmark would be needed for that claim.


In [ ]:
report=json.loads((ROOT/'reports/mp26-attention-sr.json').read_text())
for row in report['summary']: print(row)
for row in report['efficiency']: print(row)
from IPython.display import display,Image
display(Image(filename=str(ROOT/'reports/mp26-attention-sr.png')))


## Practice

1. How many tail channels would four-band x4 require?
2. Why does zero residual initialization make the model start at bicubic?
3. Does a three-seed gain on four validation scenes prove attention always helps?
4. Why can a measured model with more operations have a shorter observed latency?

## Worked solutions

1. 4×4²=64 channels before PixelShuffle.
2. The tail returns zero and the image-level skip contributes B(y).
3. No. Small-scene sampling, seed variation, the chosen degradation and model-selection protocol restrict the conclusion. Independent diverse test regions are needed.
4. Timing includes scheduling, kernel behavior and measurement noise. Compare repeated warmed measurements on identical hardware/input conditions; do not equate partial MAC counts with exact latency.

## Limits and interpretation

No new held-out test was used for this lesson. The separately frozen research result remains unchanged. The network and its average-only channel pooling differ from full published architectures. Spectral values are not calibrated reflectance, and NAIP synthetic downsampling is not an aligned satellite-pair benchmark. Full FLOPs exclude more than convolution/linear/attention MACs; process RSS is not exact allocator peak. Learner mastery is unassessed.

[Authoritative brief](README.md) · [Local/Colab setup](../../README.md)
